In [1]:
import pandas, csv, mygene, seaborn, numpy, gseapy
import scipy, scipy.stats

In [2]:
import matplotlib.pyplot
matplotlib.pyplot.rcParams.update({

    # Figure
    "figure.figsize": (7, 7),
    "figure.dpi": 300,

    # Fonts
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,

    # Axes appearance
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.5,

    # Boxplot defaults
    "boxplot.patchartist": True,
    "boxplot.showfliers": False,

    # Lines
    "lines.linewidth": 2,

    # Savefig
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

In [3]:
# 1. read results from DESeq2

results_folder = '/Users/adrian/research/bmcbf/025_isafjordur/results/002_DEGs/'
skmel_file = results_folder + 'effect_ko_vs_wt.skmel.full.tsv'
u2o2_file = results_folder + 'effect_ko_vs_wt.u2os.two.full.tsv'

res_skmel = pandas.read_csv(skmel_file, sep='\t', index_col=0)
res_u2os = pandas.read_csv(u2o2_file, sep='\t', index_col=0)

In [4]:
# 2. the genes for the cell cycle and apoptosis categories
# the file was retrieved from enrichr as reactome pathways 2024

In [5]:
def symbols_to_ensembl(
    symbols,
    species="human",
    scopes=("symbol",),
    return_all=False,
    verbose=True,
):
    """
    Convert gene symbols to Ensembl gene IDs using MyGene.

    Parameters
    ----------
    symbols : list
        List of gene symbols.
    species : str
        Species name understood by MyGene (e.g. 'human', 'mouse').
    scopes : tuple or list
        Fields to search (e.g. ('symbol',) or ('symbol', 'alias')).
    return_all : bool
        If True, return all Ensembl IDs for one-to-many mappings.
        If False, return the first Ensembl ID found.
    verbose : bool
        If True, print mapping statistics and unmapped symbols.

    Returns
    -------
    mapping : dict
        {symbol: ensembl_gene_id or None}
        If return_all=True, values are lists.
    not_found : list
        Symbols with no hit in MyGene.
    """

    mg = mygene.MyGeneInfo()

    res = mg.querymany(
        symbols,
        scopes=list(scopes),
        fields="ensembl.gene",
        species=species,
        as_dataframe=False,
    )

    mapping = {}
    not_found = []

    for r in res:
        symbol = r.get("query")

        if r.get("notfound", False):
            mapping[symbol] = None
            not_found.append(symbol)
            continue

        ensembl = r.get("ensembl")

        # No Ensembl field
        if ensembl is None:
            mapping[symbol] = None
            continue

        # Single mapping
        if isinstance(ensembl, dict):
            mapping[symbol] = (
                [ensembl.get("gene")] if return_all else ensembl.get("gene")
            )
            continue

        # Multiple mappings
        if isinstance(ensembl, list):
            genes = [
                x.get("gene")
                for x in ensembl
                if isinstance(x, dict) and x.get("gene")
            ]

            if return_all:
                mapping[symbol] = genes
            else:
                mapping[symbol] = genes[0] if len(genes) > 0 else None
            continue

        # Fallback
        mapping[symbol] = None

    if verbose:
        n_mapped = sum(v is not None and v != [] for v in mapping.values())
        print(f"Mapped: {n_mapped} / {len(mapping)}")
        if len(not_found) > 0:
            print("No hit:", not_found)

    return mapping, not_found

In [6]:
rows = []
with open('Reactome_Pathways_2024.txt', newline='') as f:
    reader = csv.reader(f, delimiter='\t')   # adjust delimiter
    for row in reader:
        rows.append(row)

df = pandas.DataFrame(rows)
mask = df[0].str.contains('TP53 Regulates', case=False, na=False)
new = df[mask]
new = new.set_index(0)
new

values = new.loc['TP53 Regulates Transcription of Cell Cycle Genes', :].dropna().to_list()
genes_cycle = [value for value in values if value != '']
print(len(genes_cycle), genes_cycle[:5])

values = new.loc['TP53 Regulates Transcription of Cell Death Genes', :].dropna().to_list()
genes_death = [value for value in values if value != '']
print(len(genes_death), genes_death[:5])

values = new.loc['TP53 Regulates Transcription of DNA Repair Genes', :].dropna().to_list()
genes_repair = [value for value in values if value != '']
print(len(genes_repair), genes_repair[:5])

# convert gene symbols into ensembl ids
output = symbols_to_ensembl(symbols=genes_cycle)
ensembl_cycle = list(output[0].values())
ensembl_cycle = [x for x in ensembl_cycle if x is not None]

output = symbols_to_ensembl(symbols=genes_death)
ensembl_death = list(output[0].values())

output = symbols_to_ensembl(symbols=genes_repair)
ensembl_repair = list(output[0].values())

# prepare data variable for analysis next
gene_sets = {
    "p53_cell_cycle": ensembl_cycle,
    "p53_apoptosis": ensembl_death,
    "p53_repair": ensembl_repair
}


Input sequence provided is already in string format. No operation performed


49 ['ARID3A', 'CNOT11', 'CNOT10', 'CDC25C', 'CNOT6']
44 ['ZNF420', 'CREBBP', 'PRELID1', 'APAF1', 'BNIP3L']
62 ['JUN', 'SUPT16H', 'GTF2F1', 'MSH2', 'GTF2F2']


2 input query terms found no hit:	['CENPJ', 'RQCD1']
Input sequence provided is already in string format. No operation performed


Mapped: 47 / 49
No hit: ['CENPJ', 'RQCD1']


Input sequence provided is already in string format. No operation performed


Mapped: 44 / 44
Mapped: 62 / 62


In [7]:
# 3. GSEA from chat

res = res_skmel

rnk = res["stat"].dropna().sort_values(ascending=False)

for k, gs in gene_sets.items():
    print(k, len(set(gs) & set(rnk.index)), "/", len(gs))


pre = gseapy.prerank(
    rnk=rnk,
    gene_sets=gene_sets,
    min_size=5,
    max_size=2000,
    permutation_num=10_000,
    seed=0,
    outdir=None,
    no_plot=True
)

pre.res2d

2026-02-16 13:56:12,394 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


p53_cell_cycle 41 / 47
p53_apoptosis 33 / 44
p53_repair 55 / 62


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,p53_apoptosis,0.78636,1.209065,0.0671,0.1382,0.131,4/33,4.10%,ENSG00000104419;ENSG00000146674;ENSG0000011512...
1,prerank,p53_cell_cycle,0.629377,0.967173,0.6125,0.92035,0.9363,18/41,28.46%,ENSG00000145632;ENSG00000118495;ENSG0000012337...
2,prerank,p53_repair,0.252091,0.389577,1.0,1.0,1.0,17/55,54.08%,ENSG00000272047;ENSG00000149136;ENSG0000009220...


In [8]:
res = res_u2os
rnk = res["stat"].dropna().sort_values(ascending=False)

for k, gs in gene_sets.items():
    print(k, len(set(gs) & set(rnk.index)), "/", len(gs))

pre = gseapy.prerank(
    rnk=rnk,
    gene_sets=gene_sets,
    min_size=5,
    max_size=2000,
    permutation_num=10_000,
    seed=0,
    outdir=None,
    no_plot=True
)
pre.res2d

2026-02-16 13:56:16,698 [WARNING] Duplicated values found in preranked stats: 0.04% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


p53_cell_cycle 45 / 47
p53_apoptosis 33 / 44
p53_repair 54 / 62


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,p53_apoptosis,0.755681,1.237668,0.0351,0.0648,0.0626,12/33,13.43%,ENSG00000115107;ENSG00000015475;ENSG0000017085...
1,prerank,p53_cell_cycle,0.611892,1.010328,0.4708,0.70515,0.845,17/45,23.40%,ENSG00000124762;ENSG00000142453;ENSG0000014911...
2,prerank,p53_repair,0.5804,0.963987,0.6302,0.620967,0.9426,9/54,15.53%,ENSG00000012048;ENSG00000100142;ENSG0000011596...


In [9]:
# 3. GSEA from chat

res = res_skmel

rnk = res["stat"].dropna().sort_values(ascending=False)

for k, gs in gene_sets.items():
    print(k, len(set(gs) & set(rnk.index)), "/", len(gs))


pre = gseapy.prerank(
    rnk=rnk,
    gene_sets=gene_sets,
    min_size=5,
    max_size=2000,
    permutation_num=10_000,
    seed=0,
    outdir=None,
    no_plot=True
)

pre.res2d

# hit_idx = pre.results[term]["hit_index"] these are the leadingedge genes leading_genes = rnk.index[hit_idx].tolist

# strict leading edge genes

es_profile = pre.results[term]["es_profile"]
peak = numpy.argmax(es_profile)

leading_idx = hit_idx[hit_idx <= peak]
leading_genes = rnk.index[leading_idx].tolist()


import gseapy.plot

term = "p53_cell_cycle"

gseapy.plot.gseaplot(
    rank_metric=pre.ranking,                 # your ranked list used by prerank
    term=term,
    **pre.results[term]                      # ES/NES/pvals + hit indices, etc.
)


import gseapy.plot

gseapy.plot.dotplot(
    pre.res2d,
    column="NES",
    title="GSEA (pre-ranked by DESeq2 stat)",
    cmap="viridis",          # you can change this; optional
    size=6
)


p53_cell_cycle 41 / 47
p53_apoptosis 33 / 44
p53_repair 55 / 62


2026-02-16 13:56:21,009 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


NameError: name 'term' is not defined

In [ ]:
p21 = 'ENSG00000124762'

s = expression.loc[p21, :]

is_skmel_ev   = s.index.str.contains(r"^SkMel28-MITFKO_ev_", regex=True)
is_skmel_mitf = s.index.str.contains(r"^SkMel28-MITFKO_mitf_x6_", regex=True)
is_u2_sc_ut   = s.index.str.contains(r"^U2-\d+_Sc_UT$", regex=True)
is_u2_simi_ut = s.index.str.contains(r"^U2-\d+_siMi_UT$", regex=True)

data = [
    s.loc[is_skmel_ev].to_numpy(),
    s.loc[is_skmel_mitf].to_numpy(),
    s.loc[is_u2_sc_ut].to_numpy(),
    s.loc[is_u2_simi_ut].to_numpy(),
]

# deseq2 was using 1 and 3 for wt and 2 and 3 for si
#data[-2] = numpy.delete(data[-2], 1) # 1 and 3 for wt
data[-1] = numpy.delete(data[-1], 0) # 2 and 3 for si

print(data)

# statistics
t_skmel = scipy.stats.ttest_ind(data[0], data[1], equal_var=False)
t_u2os  = scipy.stats.ttest_ind(data[2], data[3], equal_var=False)
print()
print("SK-MEL-28 vs siMITF SK-MEL-28:", t_skmel.pvalue)
print("U2OS vs siMITF U2OS:",         t_u2os.pvalue)
print(numpy.median(data[0]), numpy.median(data[1]))

# Boxplot
fig, ax = matplotlib.pyplot.subplots()

ax.boxplot(
    data,
    widths=0.55,
    medianprops=dict(color="0.3", zorder=4),
    boxprops=dict(facecolor="none", edgecolor="0.3", zorder=3),
    whiskerprops=dict(color="0.3", zorder=3),
    capprops=dict(color="0.3", zorder=3),
)

for i, (values, color) in enumerate(zip(data, tableau_colors), start=1):
    x_jitter = numpy.random.normal(i, 0.06, size=len(values))
    ax.scatter(
        x_jitter,
        values,
        s=90,
        alpha=0.8,
        color=color,
        edgecolor="none",
        zorder=2
    )

ax.set_xticks(range(1, 5))
ax.set_xticklabels(["SK-MEL-28", "siMITF\nSK-MEL-28", "U2OS", "siMITF\nU2OS"])

ax.set_ylabel("Expression [TPM]")
ax.set_title("CDKN1A", fontstyle="italic")

ax.tick_params(axis="x", length=6, width=1.5, color="0.3")
ax.tick_params(axis="y", length=6, width=1.5, color="0.3")
ax.yaxis.grid(True, linestyle=":", linewidth=1, color="0.5", zorder=0)

#ax.set_ylim(-25, 800)

# define ns bars
def add_ns(ax, x1, x2, y, h=0.02, text="ns"):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=2, c="k")
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=16)
add_ns(ax, 1, 2, y=-40, text='**')
add_ns(ax, 3, 4, y=-40, text='*')

ax.set_ylim(-50, 650)


matplotlib.pyplot.tight_layout()


In [ ]:
# 2. the genes for the cell cycle and apoptosis categories
# the file was retrieved from enrichr as reactome pathways 2024

rows = []
with open('Reactome_Pathways_2024.txt', newline='') as f:
    reader = csv.reader(f, delimiter='\t')   # adjust delimiter
    for row in reader:
        rows.append(row)

df = pandas.DataFrame(rows)
mask = df[0].str.contains('TP53 Regulates', case=False, na=False)
new = df[mask]
new = new.set_index(0)
new

values = new.loc['TP53 Regulates Transcription of Cell Cycle Genes', :].dropna().to_list()
genes_cycle = [value for value in values if value != '']
print(len(genes_cycle), genes_cycle[:5])

values = new.loc['TP53 Regulates Transcription of Cell Death Genes', :].dropna().to_list()
genes_death = [value for value in values if value != '']
print(len(genes_death), genes_death[:5])

values = new.loc['TP53 Regulates Transcription of DNA Repair Genes', :].dropna().to_list()
genes_repair = [value for value in values if value != '']
print(len(genes_repair), genes_repair[:5])

In [ ]:
# 3. convert gene symbols into ensembls

mg = mygene.MyGeneInfo()

mg = mygene.MyGeneInfo()
out = mg.querymany(genes_cycle,
                   scopes='symbol',
                   fields='ensembl.gene',
                   species='human',
                   as_dataframe=True, return_all=True)

print(out['ensembl.gene'].to_list()[:50])
print(out['ensembl'].to_list()[:50])

missing = []
nan_rows = out[pandas.isna(out['ensembl.gene'])]
for idx, row in nan_rows.iterrows():
    if isinstance(row['ensembl'], float) == False:
        for element in row['ensembl']:
            print(element)
        missing.append(element['gene'])
    print()
print(len(missing))
all_ids = out['ensembl.gene'].to_list() + missing
print(len(all_ids), all_ids[:50])
unique = list(set(all_ids))
ensembl_repair = list(filter(pandas.notna, unique))
print(len(ensembl_repair), ensembl_repair[:50])

out





In [ ]:
# make a heatmap

In [ ]:
mini = expression[expression.index.isin(ensembl_repair)]
print(mini.shape)

#mini = mini[mini.max(axis=1) > 2]
#print(mini.shape)

#mini = sub[(sub.max(axis=1) - sub.min(axis=1)) > 20]
#print(mini.shape)

df_z = mini.sub(mini.mean(axis=1), axis=0).div(mini.std(axis=1), axis=0)
print(df_z.shape)

In [ ]:
mini

In [ ]:
map_df = mg.querymany(df_z.index,
                   scopes='ensembl.gene',
                   fields='symbol',
                   species='human',
                   as_dataframe=True, return_all=True)

df_z = df_z.rename(index=map_df["symbol"])


In [ ]:
df_z

In [ ]:
seaborn.set(style="white")          # aesthetic style (optional)

g = seaborn.clustermap(
    df_z,
    method='average',           # linkage method: 'single', 'complete', 'ward', …
    metric='euclidean',         # distance metric
    cmap='bwr',             # colour map for the heat‑map
    figsize=(12, 8),
    linewidths=.5,
    linecolor='gray',
    dendrogram_ratio=(.2, .2),  # proportion of figure devoted to dendrograms
    cbar_pos=(0.02, .2, .03, .4), # position of colour bar (optional)
    vmin=-2,
    yticklabels=True,
    vmax= 2
)
g.ax_heatmap.tick_params(axis="y", labelsize=8)
